# 04. Entraînement final et évaluation sur les ventes de 2012

Objectif : entraîner la configuration retenue sur toutes les ventes antérieures à 2012, puis mesurer sa performance sur les quatre premiers mois de 2012.

**Choix arrêté avant cette évaluation : méthode A et un seul CatBoost global**, avec `ProductGroup` parmi les variables catégorielles. Le score de test ne sert donc pas à choisir : il mesure une décision déjà prise.

Les scores affichés proviennent de l'exécution du 2026-09-09 ; aucun chiffre n'est saisi à la main. Exécuter les cellules dans l'ordre. Ce notebook peut être lancé avec un noyau neuf, sans réexécuter les expériences de 03.

## Choix final : précision et maintenabilité

Les résultats de `03 - Synthese des experiences`, enregistrés dans `resultats/20260909_181245_359760`, favorisent la segmentation :

| Validation | Global A | Segmenté A | Réduction relative du RMSLE |
|---|---:|---:|---:|
| Septembre–décembre 2011 | 0,23351 | 0,21740 | 6,9 % |
| Janvier–avril 2011 | 0,24370 | 0,23050 | 5,4 % |

Le gain est observé sur les six familles dans les deux périodes. La segmentation satisfait les critères prédictifs fixés dans 03 ; elle reste le meilleur candidat mesuré avec ce budget.

**Arbitrage d'exploitation décidé ensuite, avant le test 2012 :** nous retenons le global pour une première version destinée à la production. Il ne comporte qu'un modèle à versionner, réentraîner et déployer, contre six spécialistes et un global de secours pour l'autre solution. Le suivi des performances par famille reste nécessaire dans les deux cas.

La réduction du RMSLE ne se traduit pas directement en économies financières. Le bénéfice métier et les coûts d'exploitation supplémentaires n'ont pas été mesurés. Le choix du global est donc un compromis assumé de simplicité, pas une supériorité prédictive ni une validation complète de son aptitude à la production.

Cette décision ne remplace pas les résultats historiques de 03 et ne prétend pas découler de ses seuls seuils prédictifs. La segmentation reste une alternative documentée, à reconsidérer si le bénéfice métier justifie sa complexité.

**Pourquoi A ?** Pour le global, B améliore le RMSLE de seulement 0,00093 sur la validation principale, sous le seuil pratique de 0,002. On conserve donc le traitement catégoriel `Missing` de A et les imputations numériques prévues.

## Un seul modèle, avec les catégories natives

`ProductGroup` est donné à CatBoost dans `cat_features`, avec les autres variables catégorielles. Le modèle apprend sur toutes les familles et peut exploiter leurs différences dans ses arbres. Aucun routage vers des spécialistes n'est créé.

La préparation est celle de `src.preparation_03.Preparation03`, déjà vérifiée dans 03. On la réajuste entièrement sur les ventes antérieures à 2012 : médianes, fréquences et regroupements. Le test reçoit uniquement `transform`.

La cible reste `log1p(SalePrice)` ; les prédictions de CatBoost sont donc en log. `np.expm1` permet ensuite de revenir au prix.

In [1]:
from pathlib import Path
import sys
import json
import pickle
import numpy as np
import pandas as pd
from catboost import CatBoostRegressor
from sklearn.metrics import mean_squared_error
from IPython.display import display

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents]
            if (p / "src/preparation_03.py").exists())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from src.preparation_03 import Preparation03

# Passer à False pour recharger le modèle déjà entraîné au lieu de le réentraîner.
REENTRAINER = True

PARAMS = dict(iterations=500, depth=6, learning_rate=0.1,
              loss_function="RMSE", random_seed=2012,
              thread_count=8,  # fixé : rend les durées comparables d'une exécution à l'autre
              verbose=100, allow_writing_files=False)
RAW = ROOT / "data/raw/bluebook-for-bulldozers/TrainAndValid.csv"
DOSSIER_FINAL = ROOT / "resultats/modele_final_global_A"
DOSSIER_FINAL.mkdir(parents=True, exist_ok=True)
FICHIER_MODELE = DOSSIER_FINAL / "modele_final.pkl"

## Données d'apprentissage

Le fichier brut regroupe les ventes anciennes et celles de 2012. Seules les lignes antérieures au 1er janvier 2012 sont utilisées ici. Les anciennes fenêtres de validation rejoignent maintenant l'apprentissage, puisque le choix est arrêté.

In [2]:
df = pd.read_csv(RAW, low_memory=False)
# Format imposé : on ne laisse pas pandas inférer un format américain MM/DD/YYYY.
df["saledate"] = pd.to_datetime(df["saledate"], format="%m/%d/%Y %H:%M")
train_final = df.loc[df["saledate"] < "2012-01-01"].sort_values("saledate", kind="stable").copy()
assert train_final["SalePrice"].gt(0).all(), "Prix nul ou négatif dans l'apprentissage"

y_train_final = np.log1p(train_final["SalePrice"])
print(f"Apprentissage final : {len(train_final):,} ventes")
print(train_final["saledate"].min(), "→", train_final["saledate"].max())

Apprentissage final : 401,125 ventes
1989-01-17 00:00:00 → 2011-12-30 00:00:00


## Préparation et entraînement final

On conserve les paramètres testés dans 03. Il n'y a ni recherche d'hyperparamètres ni arrêt anticipé à partir du test.

Choix assumé plutôt qu'un oubli : une recherche systématique (grille, RandomizedSearchCV) multiplierait le coût de calcul déjà élevé (~10 min pour le seul modèle global sur 500 itérations, x plusieurs spécialistes par famille), pour un gain probablement inférieur au seuil de bruit du projet (0,002) ; le levier qui a fait la différence dans cette étude est le choix d'architecture (segmentation), pas le réglage fin des hyperparamètres.

La préparation est réajustée entièrement sur les ventes antérieures à 2012 (médianes, fréquences et regroupements), puis le modèle est entraîné et sauvegardé avec elle dans un fichier unique. Pour rouvrir le notebook sans refaire les douze minutes d'entraînement, passer `REENTRAINER = False` en tête.

In [3]:
if REENTRAINER:
    preparation_finale = Preparation03(methode="A")
    X_train_final = preparation_finale.fit_transform(train_final)
    cat_features = preparation_finale.cat_features_
    assert "ProductGroup" in cat_features
    assert not {"SalePrice", "logSalePrice", "saledate"}.intersection(X_train_final.columns)

    modele_final = CatBoostRegressor(**PARAMS)
    modele_final.fit(X_train_final, y_train_final, cat_features=cat_features)

    with FICHIER_MODELE.open("wb") as fichier:
        pickle.dump({"modele": modele_final, "preparation": preparation_finale}, fichier)
    print("Modèle final enregistré :", FICHIER_MODELE)
else:
    with FICHIER_MODELE.open("rb") as fichier:
        sauvegarde = pickle.load(fichier)
    preparation_finale = sauvegarde["preparation"]
    modele_final = sauvegarde["modele"]
    print("Modèle final et préparation rechargés :", FICHIER_MODELE)

# Garde-fou principal : la préparation ne doit avoir vu aucune vente de 2012.
assert preparation_finale.fit_max_date_ < pd.Timestamp("2012-01-01")
print("Un seul CatBoost ; ProductGroup fait partie des variables catégorielles.")

0:	learn: 0.6496923	total: 1.48s	remaining: 12m 17s
100:	learn: 0.2534212	total: 2m 27s	remaining: 9m 43s
200:	learn: 0.2363891	total: 5m 25s	remaining: 8m 3s
300:	learn: 0.2278731	total: 9m 8s	remaining: 6m 2s
400:	learn: 0.2226014	total: 12m 15s	remaining: 3m 1s
499:	learn: 0.2191278	total: 14m 32s	remaining: 0us
Modèle final enregistré : c:\Users\Abram\Desktop\Fiche\Projet\resultats\modele_final_global_A\modele_final.pkl
Un seul CatBoost ; ProductGroup fait partie des variables catégorielles.


## Évaluation sur le test 2012

Cette étape mesure la performance de la configuration déjà choisie. Le score ne doit pas servir à revenir sur A/B, à sélectionner l'architecture ou à régler les paramètres.

Le test 2012 a déjà été consulté dans d'anciennes versions du projet, avant la correction du protocole. Cette exécution est l'évaluation finale de la configuration actuelle, mais le jeu n'est pas un échantillon totalement inédit dans l'histoire du projet : une validation indépendante ultérieure resterait utile.

Le score, les prédictions et le détail par famille sont enregistrés dans `resultats/modele_final_global_A`.

In [4]:
test_final = df.loc[df["saledate"] >= "2012-01-01"].sort_values("saledate", kind="stable").copy()
assert train_final["saledate"].max() < test_final["saledate"].min(), "Chevauchement apprentissage/test"
assert train_final.index.intersection(test_final.index).empty, "Lignes communes apprentissage/test"
assert test_final["SalePrice"].gt(0).all()

y_test_final = np.log1p(test_final["SalePrice"])
X_test_final = preparation_finale.transform(test_final)
pred_test_log = modele_final.predict(X_test_final)
rmse_test = mean_squared_error(y_test_final, pred_test_log) ** 0.5

predictions_test = pd.DataFrame({"SalesID": test_final["SalesID"],
                                 "saledate": test_final["saledate"],
                                 "ProductGroup": test_final["ProductGroup"],
                                 "prix_reel": test_final["SalePrice"],
                                 "prix_predit": np.expm1(pred_test_log),
                                 "log_prix_reel": y_test_final,
                                 "log_prix_predit": pred_test_log})
resultats_segments = []
for groupe, ventes in predictions_test.groupby("ProductGroup", dropna=False):
    score = mean_squared_error(ventes["log_prix_reel"], ventes["log_prix_predit"]) ** 0.5
    resultats_segments.append({"segment": str(groupe), "n": len(ventes), "rmsle": float(score)})

rapport_test = {"methode": "A", "architecture": "global", "n_train": len(train_final),
                "n_test": len(test_final), "rmsle": float(rmse_test),
                "debut_test": str(test_final["saledate"].min()),
                "fin_test": str(test_final["saledate"].max()),
                "par_segment": resultats_segments,
                "limite": "Le test 2012 a été consulté dans des versions historiques du projet."}
predictions_test.to_csv(DOSSIER_FINAL / "predictions_test.csv", index=False)
(DOSSIER_FINAL / "evaluation_test.json").write_text(
    json.dumps(rapport_test, ensure_ascii=False, indent=2), encoding="utf-8")

print(f"RMSLE test 2012 : {rmse_test:.5f} — {len(test_final):,} ventes")
display(pd.DataFrame(resultats_segments))

RMSLE test 2012 : 0.24043 — 11,573 ventes


,segment,n,rmsle
0,BL,1986,0.195450
1,MG,769,0.240657
2,SSL,1523,0.222945
3,TEX,3063,0.251216
4,TTT,2062,0.260263
5,WL,2170,0.253900


## Utiliser le modèle sur de nouvelles ventes

On transmet les caractéristiques brutes, y compris `saledate` et `ProductGroup`, mais pas le prix à prédire. La préparation sauvegardée applique les règles déjà apprises ; on ne relance pas `fit`.

L'exemple utilise trois ventes d'apprentissage uniquement pour vérifier l'appel de prédiction. Ce n'est pas une nouvelle mesure de performance.

In [5]:
nouvelles_ventes = train_final.head(3).drop(columns=["SalePrice"])
X_nouvelles_ventes = preparation_finale.transform(nouvelles_ventes)
predictions_prix = np.expm1(modele_final.predict(X_nouvelles_ventes))
display(pd.DataFrame({"SalesID": nouvelles_ventes["SalesID"], "prix_predit": predictions_prix}))

,SalesID,prix_predit
205615,1646770,11464.160657
28603,1259551,9511.085922
28757,1259760,10573.461695


## Lecture du résultat

Le RMSLE global décrit l'erreur sur toutes les ventes du test ; ce n'est pas une moyenne simple des scores par famille. Un RMSLE de 0,23 ne signifie pas directement « 23 % d'erreur ».

Le tableau par famille sert à repérer les limites du modèle unique. Il ne sert pas à remplacer après coup certaines prédictions par celles de spécialistes sur ce même test. Les prochaines évolutions demanderont de nouvelles validations temporelles.

Le dossier `resultats/modele_final_global_A` contient la préparation et le modèle, le rapport de test et les prédictions. Les prochaines étapes de production restent le suivi de dérive, la politique de réentraînement et le mode de déploiement.